# PyTorch 第八章：图像项目案例

> 对应《PyTorch 实用教程（第二版）》第八章  
> 目标：用**最小可运行实验**理解分类、分割、检测、跟踪、GAN、DDPM、Image Captioning、CLIP 与图像检索的核心机制。

## 本章覆盖顺序

1. 8.1 图像分类——胸部 X 光肺炎分类  
2. 8.2 图像分割——脑 MRI 胶质瘤分割  
3. 8.3 目标检测——无人机检测  
4. 8.4 目标跟踪（上）——DeepSORT 原理  
5. 8.4 目标跟踪（下）——车流量统计  
6. 8.5 生成对抗网络——CycleGAN  
7. 8.6 扩散模型——DDPM  
8. 8.7 图像描述——Image Captioning  
9. 8.8 图像检索（上）——理论基础  
10. 8.8 图像检索（下）——CLIP + Faiss + Web 服务

原教程章节入口：  
https://tingsongyu.github.io/PyTorch-Tutorial-2nd/chapter-8/

### 本 Notebook 的取舍

- 不下载大型数据集、YOLO/CLIP/GPT-2 权重。
- 不复制大型工程源码，只保留数据流、张量形状、损失与模块关系。
- 代码全部基于小型随机/模拟数据，可在 Google Colab 直接运行。
- 对旧接口按当前 PyTorch/torchvision 写法更新。

## 0. 环境导入

本章重点不是训练出高精度模型，而是理解不同视觉任务的**输入、输出、损失、匹配与检索机制**。

In [ ]:
import math
import random
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("device:", device)

### 当前 PyTorch 写法提醒

- 旧教程常见：`resnet50(pretrained=True)`  
  当前 torchvision 推荐：`resnet50(weights=ResNet50_Weights.DEFAULT)`。
- 推理阶段优先使用 `torch.inference_mode()`。
- AMP 新接口使用 `torch.amp.autocast(...)` / `torch.amp.GradScaler(...)`，不再推荐旧的 `torch.cuda.amp.*`。
- 本 Notebook 为避免下载权重，不实际加载 ImageNet 参数。

当前接口示例：

```python
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)
preprocess = weights.transforms()
```

# 8.1 图像分类——胸部 X 光肺炎分类

原教程顺序：**数据模块 → 数据增强 → 模型模块 → 训练实验 → 推理与测速**。

分类任务最核心的张量关系：

- 输入：$x \in \mathbb{R}^{N\times C\times H\times W}$
- 输出 logits：$z \in \mathbb{R}^{N\times K}$
- 标签：$y \in \{0,\dots,K-1\}^N$
- 多分类常用损失：`CrossEntropyLoss`

胸片是灰度图，但使用 ImageNet 预训练模型时，经常会复制为 3 通道，或修改第一层卷积。

In [ ]:
# 模拟灰度胸片：8 个样本，1 通道，64x64
x_cls = torch.randn(8, 1, 64, 64)
y_cls = torch.randint(0, 2, (8,))

print("x:", x_cls.shape)
print("y:", y_cls.shape, y_cls.dtype)

## 数据增强：医学图像不能机械套用自然图像策略

原教程比较了手工增强与 AutoAugment。关键不是“增强越多越好”，而是增强是否保留医学语义。

例如：某些旋转、裁剪、颜色扰动在自然图像中合理，但在医学影像中可能改变诊断信息。

In [ ]:
# 最小可运行增强：随机水平翻转
def random_hflip(x, p=0.5):
    if torch.rand(()) < p:
        return torch.flip(x, dims=[-1])
    return x

x_aug = random_hflip(x_cls.clone(), p=1.0)
print("shape unchanged:", x_aug.shape)
print("pixels changed:", not torch.equal(x_aug, x_cls))

## 模型：修改分类头

预训练 CNN 的核心复用方式是：

1. backbone 提取视觉特征；
2. 替换最后的分类层；
3. 根据任务决定冻结 backbone 还是整体微调。

这里用小模型等价演示。

In [ ]:
class TinyClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(16, num_classes)

    def forward(self, x):
        feat = self.features(x).flatten(1)
        return self.classifier(feat)

model_cls = TinyClassifier().to(device)
logits = model_cls(x_cls.to(device))
loss_cls = F.cross_entropy(logits, y_cls.to(device))

print("logits:", logits.shape)
print("loss:", float(loss_cls))

In [ ]:
optimizer = torch.optim.AdamW(model_cls.parameters(), lr=1e-3)

optimizer.zero_grad(set_to_none=True)
logits = model_cls(x_cls.to(device))
loss = F.cross_entropy(logits, y_cls.to(device))
loss.backward()
optimizer.step()

grad_norm = model_cls.classifier.weight.grad.norm().item()
print("classifier grad norm:", round(grad_norm, 6))

## 推理：`eval()` 与 `inference_mode()` 是两件事

- `model.eval()`：切换 Dropout / BatchNorm 行为。
- `torch.inference_mode()`：关闭 autograd，并减少推理期开销。

In [ ]:
model_cls.eval()
with torch.inference_mode():
    pred = model_cls(x_cls.to(device)).argmax(dim=1)

print("pred:", pred.cpu().tolist())

# 8.2 图像分割——脑 MRI 胶质瘤分割

原教程顺序：**数据划分 → Dataset → 训练 → SMP → 对比实验 → 推理**。

最重要的工程原则不是网络，而是：

> 同一患者的连续切片不能同时出现在训练集和验证集，否则会发生数据泄漏。

二分类语义分割常见形式：

- 模型输出 logits：`[N, 1, H, W]`
- mask：`[N, 1, H, W]`
- `BCEWithLogitsLoss`：target 通常为 float 的 0/1
- 多分类 `CrossEntropyLoss`：target 为 `long`，形状 `[N,H,W]`

这比“所有分割标签都必须 long”更准确。

In [ ]:
# 模拟 6 位患者，每人 4 张切片
patient_ids = np.repeat(np.arange(6), 4)
indices = np.arange(len(patient_ids))

train_patients = {0, 1, 2, 3}
train_idx = indices[np.isin(patient_ids, list(train_patients))]
val_idx = indices[~np.isin(patient_ids, list(train_patients))]

print("train patients:", sorted(set(patient_ids[train_idx])))
print("val patients:", sorted(set(patient_ids[val_idx])))
print("patient overlap:", set(patient_ids[train_idx]) & set(patient_ids[val_idx]))

## Dice 与 IoU

对于前景集合 $A$ 与标签集合 $B$：

$$
Dice=rac{2|A\cap B|}{|A|+|B|}
$$

$$
IoU=rac{|A\cap B|}{|A\cup B|}
$$

Dice 对小目标更常见，IoU 更直观。二者都必须先明确“按图统计”还是“跨整个数据集累计统计”。

In [ ]:
def dice_score_from_logits(logits, target, threshold=0.5, eps=1e-6):
    pred = (logits.sigmoid() >= threshold).float()
    dims = tuple(range(1, pred.ndim))
    inter = (pred * target).sum(dim=dims)
    denom = pred.sum(dim=dims) + target.sum(dim=dims)
    return ((2 * inter + eps) / (denom + eps)).mean()

def iou_score_from_logits(logits, target, threshold=0.5, eps=1e-6):
    pred = (logits.sigmoid() >= threshold).float()
    dims = tuple(range(1, pred.ndim))
    inter = (pred * target).sum(dim=dims)
    union = pred.sum(dim=dims) + target.sum(dim=dims) - inter
    return ((inter + eps) / (union + eps)).mean()

seg_logits = torch.randn(2, 1, 32, 32)
seg_target = (torch.rand(2, 1, 32, 32) > 0.8).float()

bce = F.binary_cross_entropy_with_logits(seg_logits, seg_target)
print("BCE:", round(float(bce), 4))
print("Dice:", round(float(dice_score_from_logits(seg_logits, seg_target)), 4))
print("IoU:", round(float(iou_score_from_logits(seg_logits, seg_target)), 4))

## Encoder-Decoder：分割模型的核心结构

U-Net / FPN / DeepLab 的差异很多，但共同问题都是：

1. encoder 压缩空间、提取语义；
2. decoder 恢复空间分辨率；
3. 跳连保留低层细节；
4. 输出必须与 mask 空间尺寸对齐。

In [ ]:
class TinyUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(),
            nn.Conv2d(8, 8, 3, padding=1), nn.ReLU()
        )
        self.down = nn.MaxPool2d(2)
        self.mid = nn.Sequential(
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU()
        )
        self.up = nn.ConvTranspose2d(16, 8, kernel_size=2, stride=2)
        self.out = nn.Conv2d(16, 1, kernel_size=1)

    def forward(self, x):
        skip = self.enc(x)
        h = self.mid(self.down(skip))
        h = self.up(h)
        h = torch.cat([h, skip], dim=1)
        return self.out(h)

u = TinyUNet()
x = torch.randn(2, 1, 64, 64)
out = u(x)
print("input:", x.shape)
print("output:", out.shape)

# 8.3 目标检测——无人机检测

原教程顺序：**数据格式 → 标签转换 → 检测框架 → YOLO 训练机制 → 对比实验 → 推理**。

常见 bbox 格式：

- VOC：`(xmin, ymin, xmax, ymax)`
- COCO：`(xmin, ymin, width, height)`
- YOLO：归一化 `(cx, cy, width, height)`

最容易错的是：**坐标格式、是否归一化、宽高对应哪个图像尺寸**。

In [ ]:
def yolo_to_xyxy(box, img_w, img_h):
    cx, cy, w, h = box
    cx, w = cx * img_w, w * img_w
    cy, h = cy * img_h, h * img_h
    return torch.tensor([cx - w/2, cy - h/2, cx + w/2, cy + h/2])

box_yolo = torch.tensor([0.5, 0.5, 0.4, 0.2])
box_xyxy = yolo_to_xyxy(box_yolo, img_w=640, img_h=480)

print("YOLO:", box_yolo.tolist())
print("xyxy:", box_xyxy.tolist())

## IoU 与 NMS

检测器会产生大量重叠候选框。NMS（Non-Maximum Suppression）保留高置信度框，并抑制与其 IoU 过大的低分框。

$$
IoU(A,B)=\frac{|A\cap B|}{|A\cup B|}
$$

In [ ]:
def box_iou_one_to_many(box, boxes):
    x1 = torch.maximum(box[0], boxes[:, 0])
    y1 = torch.maximum(box[1], boxes[:, 1])
    x2 = torch.minimum(box[2], boxes[:, 2])
    y2 = torch.minimum(box[3], boxes[:, 3])

    inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
    area1 = (box[2] - box[0]) * (box[3] - box[1])
    area2 = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return inter / (area1 + area2 - inter + 1e-6)

def nms_pytorch(boxes, scores, iou_threshold=0.5):
    order = scores.argsort(descending=True)
    keep = []
    while order.numel() > 0:
        i = order[0].item()
        keep.append(i)
        if order.numel() == 1:
            break
        iou = box_iou_one_to_many(boxes[i], boxes[order[1:]])
        order = order[1:][iou <= iou_threshold]
    return torch.tensor(keep, dtype=torch.long)

boxes = torch.tensor([
    [10., 10., 50., 50.],
    [12., 12., 48., 48.],
    [60., 60., 95., 95.]
])
scores = torch.tensor([0.95, 0.80, 0.90])

keep = nms_pytorch(boxes, scores, 0.5)
print("keep indices:", keep.tolist())

当前 torchvision 已直接提供 `torchvision.ops.nms` / `batched_nms`。真实项目中应优先使用官方实现，而不是手写循环。

YOLO 工程虽然复杂，但训练骨架仍然是：

`Dataset/DataLoader → nn.Module → loss → optimizer → scheduler → validation → NMS`

# 8.4 目标跟踪（上）——DeepSORT 原理

原教程先从 SORT 的问题出发，再进入 DeepSORT：

1. 检测框与历史轨迹如何匹配？
2. 漏检时为什么不能立刻删除轨迹？
3. 误检时为什么不能立刻确认新轨迹？
4. 如何结合**运动信息**与**外观特征**？

DeepSORT 的核心不是“再训练一个检测器”，而是**数据关联（data association）**。

## 匹配 = 构造代价矩阵 + 求最优分配

代价可来自：

- appearance cosine distance
- Mahalanobis distance
- IoU distance

随后用 Hungarian Algorithm 求最小总代价匹配。

In [ ]:
from scipy.optimize import linear_sum_assignment

# 3 个轨迹 vs 3 个检测框的代价
cost = np.array([
    [0.10, 0.80, 0.70],
    [0.75, 0.20, 0.65],
    [0.60, 0.55, 0.15],
])

rows, cols = linear_sum_assignment(cost)
pairs = list(zip(rows.tolist(), cols.tolist()))

print("matches:", pairs)
print("total cost:", cost[rows, cols].sum())

## 卡尔曼滤波：预测 + 观测融合

DeepSORT 中的运动模型维护状态均值和协方差。直观上：

$$
\text{new estimate}
=
\text{prediction}
+
K(\text{measurement}-\text{prediction})
$$

$K$ 是 Kalman Gain：观测越可信，越相信当前检测；预测越可信，越相信历史运动模型。

In [ ]:
# 1D 位置 + 速度的极简 Kalman Filter
x = np.array([[0.0], [1.0]])  # position, velocity
P = np.eye(2)
Fmat = np.array([[1.0, 1.0],
                 [0.0, 1.0]])
H = np.array([[1.0, 0.0]])
Q = np.eye(2) * 0.01
R = np.array([[0.25]])

measurement = np.array([[1.2]])

# predict
x_pred = Fmat @ x
P_pred = Fmat @ P @ Fmat.T + Q

# update
innovation = measurement - H @ x_pred
S = H @ P_pred @ H.T + R
K = P_pred @ H.T @ np.linalg.inv(S)
x_new = x_pred + K @ innovation
P_new = (np.eye(2) - K @ H) @ P_pred

print("predicted position:", round(float(x_pred[0, 0]), 3))
print("measurement:", float(measurement[0, 0]))
print("updated position:", round(float(x_new[0, 0]), 3))

## 轨迹状态机

DeepSORT 常见思想：

- Tentative：新轨迹先观察若干帧；
- Confirmed：连续匹配后确认；
- Deleted：连续多帧未匹配后删除。

目的：过滤短暂误检，并允许短暂漏检。

# 8.4 目标跟踪（下）——车流量统计

原教程顺序：**DeepSORT 类结构 → YOLO + DeepSORT → 撞线/区域计数 → 工程注意事项**。

核心抽象：

- `Track`：单个目标的 id、bbox、状态、外观特征；
- `Tracker`：管理所有轨迹并完成匹配、预测、更新；
- `KalmanFilter`：运动预测；
- `DeepSort`：对外提供统一 `update()` 接口。

## 区域计数：本质是有限状态变化

例如定义：

- outer 区域
- inner 区域

同一个 `track_id` 先到 outer 再到 inner，则计一次“进入”；反方向则计一次“离开”。

In [ ]:
class TwoRegionCounter:
    def __init__(self):
        self.last_region = {}
        self.enter = 0
        self.exit = 0

    def update(self, track_id, region):
        prev = self.last_region.get(track_id)
        if prev == "outer" and region == "inner":
            self.enter += 1
        elif prev == "inner" and region == "outer":
            self.exit += 1
        self.last_region[track_id] = region

counter = TwoRegionCounter()
events = [(7, "outer"), (7, "inner"), (9, "inner"), (9, "outer")]

for tid, region in events:
    counter.update(tid, region)

print("enter:", counter.enter)
print("exit:", counter.exit)

### 工程陷阱：mask 不能随便插值

原教程指出：若区域 mask 用整数 1/2 表示类别，普通 resize 插值可能制造不存在的中间/错误边界。

离散标签图缩放应使用 **nearest-neighbor**，不要用双线性插值。

# 8.5 生成对抗网络——CycleGAN

原教程顺序：**GAN → CycleGAN 结构与损失 → 训练注意事项 → 数据 → 模型 → 推理**。

CycleGAN 的关键是：**不需要成对图片（unpaired data）**。

设：

- $G:A\rightarrow B$
- $F:B\rightarrow A$
- $D_A,D_B$：两个判别器

主要损失：

1. adversarial loss
2. cycle consistency loss
3. identity loss

## Cycle Consistency

如果 $G$ 把 A 域图像变成 B 域，那么再用 $F$ 应尽量还原：

$$
F(G(x_A))\approx x_A
$$

因此：

$$
L_{cyc}=||F(G(x_A))-x_A||_1 + ||G(F(x_B))-x_B||_1
$$

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c, c, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(c, c, 3, padding=1),
        )
    def forward(self, x):
        return x + self.net(x)

class TinyGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            ResBlock(16),
            nn.Conv2d(16, 3, 3, padding=1),
            nn.Tanh(),
        )
    def forward(self, x):
        return self.net(x)

class TinyPatchDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(16, 1, 4, stride=2, padding=1),
        )
    def forward(self, x):
        return self.net(x)

G = TinyGenerator()
Fgen = TinyGenerator()
D_A = TinyPatchDiscriminator()
D_B = TinyPatchDiscriminator()

real_A = torch.randn(2, 3, 32, 32).clamp(-1, 1)
real_B = torch.randn(2, 3, 32, 32).clamp(-1, 1)

fake_B = G(real_A)
fake_A = Fgen(real_B)
rec_A = Fgen(fake_B)
rec_B = G(fake_A)

print("fake_B:", fake_B.shape)
print("patch logits:", D_B(fake_B).shape)

In [ ]:
mse = nn.MSELoss()
l1 = nn.L1Loss()

# LSGAN：生成器希望判别器把 fake 判为 1
loss_gan = mse(D_B(fake_B), torch.ones_like(D_B(fake_B)))
loss_cycle = l1(rec_A, real_A) + l1(rec_B, real_B)

# identity：B 输入 G 时尽量保持 B；A 输入 F 时尽量保持 A
loss_id = l1(G(real_B), real_B) + l1(Fgen(real_A), real_A)

loss_G = loss_gan + 10.0 * loss_cycle + 0.5 * loss_id
loss_G.backward()

print("GAN:", round(float(loss_gan), 4))
print("cycle:", round(float(loss_cycle), 4))
print("identity:", round(float(loss_id), 4))

注意：GAN 的 G/D loss 往往会振荡，不能仅凭 loss 曲线判断视觉质量。实际训练还要观察样张、模式崩溃（mode collapse）和域间可转换性。

# 8.6 扩散模型——DDPM

原教程顺序：**Diffusion 简介 → 加噪/去噪步骤 → 公式 → U-Net 结构 → 训练 → 推理 → guidance → Stable Diffusion / LDM**。

DDPM 训练最关键的重参数公式：

$$
x_t=\sqrt{\bar{\alpha}_t}x_0+\sqrt{1-\bar{\alpha}_t}\epsilon,
\quad \epsilon\sim\mathcal N(0,I)
$$

模型输入：

- noisy image $x_t$
- timestep $t$

模型学习预测噪声 $\epsilon$。

In [ ]:
T = 100
betas = torch.linspace(1e-4, 2e-2, T)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

def q_sample(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    a_bar = alpha_bars[t].view(-1, 1, 1, 1).to(x0.device)
    xt = a_bar.sqrt() * x0 + (1 - a_bar).sqrt() * noise
    return xt, noise

x0 = torch.randn(4, 3, 16, 16)
t = torch.tensor([0, 10, 50, 99])
xt, eps = q_sample(x0, t)

print("x0:", x0.shape)
print("xt:", xt.shape)
print("noise std:", round(float(eps.std()), 3))

## 为什么训练时不用真的逐步加 1000 次噪声？

因为闭式公式可以直接由 $x_0$ 得到任意 $x_t$。

所以每个 batch 只需：

1. 随机采样 timestep；
2. 随机采样高斯噪声；
3. 一步构造 $x_t$；
4. 让网络预测这份噪声。

In [ ]:
class TinyNoisePredictor(nn.Module):
    def __init__(self, T=100, hidden=32):
        super().__init__()
        self.time_emb = nn.Embedding(T, hidden)
        self.time_proj = nn.Linear(hidden, hidden)
        self.in_conv = nn.Conv2d(3, hidden, 3, padding=1)
        self.mid = nn.Sequential(
            nn.GroupNorm(4, hidden),
            nn.SiLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.SiLU(),
        )
        self.out = nn.Conv2d(hidden, 3, 3, padding=1)

    def forward(self, x, t):
        h = self.in_conv(x)
        temb = self.time_proj(self.time_emb(t))[:, :, None, None]
        h = h + temb
        h = self.mid(h)
        return self.out(h)

noise_model = TinyNoisePredictor(T)
opt = torch.optim.AdamW(noise_model.parameters(), lr=1e-3)

x0 = torch.randn(8, 3, 16, 16)
t = torch.randint(0, T, (8,))
xt, noise = q_sample(x0, t)

pred_noise = noise_model(xt, t)
loss_ddpm = F.mse_loss(pred_noise, noise)

opt.zero_grad(set_to_none=True)
loss_ddpm.backward()
opt.step()

print("pred_noise:", pred_noise.shape)
print("DDPM MSE:", round(float(loss_ddpm), 4))

## U-Net 中为什么要加入 timestep embedding？

同一张图在 $t=20$ 与 $t=900$ 时噪声强度完全不同。模型必须知道当前处于哪一个扩散阶段。

现代扩散 U-Net 通常还包含：

- ResBlock
- timestep embedding
- self-attention / cross-attention
- skip connection

### Guidance

- classifier guidance：额外分类器提供梯度方向；
- classifier-free guidance：训练时同时学习 conditional / unconditional；
- 文本条件经过 embedding 后进入 cross-attention，是 Stable Diffusion 一类模型的重要机制。

### Latent Diffusion

Stable Diffusion 不直接在高分辨率 RGB 像素空间扩散，而是在 VAE latent 中扩散，显著降低计算量。

# 8.7 图像描述——Image Captioning

原教程顺序：**概念与发展 → BLEU → CNN+RNN+Attention → CLIPCap**。

任务形式：

$$
\text{image}\rightarrow \text{caption tokens}
$$

本节是后续理解多模态 LLM 的重要桥梁。

## BLEU：不要只记“n-gram 命中率”

标准 BLEU 还包含：

- clipped n-gram precision
- brevity penalty
- 多阶 n-gram 几何平均

下面只演示最核心的 clipped precision。

In [ ]:
def ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def clipped_precision(candidate, reference, n):
    c = Counter(ngrams(candidate, n))
    r = Counter(ngrams(reference, n))
    matched = sum(min(v, r[g]) for g, v in c.items())
    total = max(sum(c.values()), 1)
    return matched / total

cand = "the cat sat on the mat".split()
ref = "the cat is on the mat".split()

for n in range(1, 5):
    print(f"{n}-gram precision:", round(clipped_precision(cand, ref, n), 3))

## CNN + RNN + Attention

经典数据流：

`image → CNN feature map → attention → context vector → LSTM → next-token logits`

Attention 的关键作用：生成不同单词时，动态关注不同图像区域。

In [ ]:
B, L, D_img = 2, 16, 32   # 16 个图像区域
D_hidden = 24

image_features = torch.randn(B, L, D_img)
hidden = torch.randn(B, D_hidden)

attn_img = nn.Linear(D_img, D_hidden, bias=False)
attn_h = nn.Linear(D_hidden, D_hidden, bias=False)
attn_score = nn.Linear(D_hidden, 1, bias=False)

scores = attn_score(torch.tanh(
    attn_img(image_features) + attn_h(hidden).unsqueeze(1)
)).squeeze(-1)

weights = scores.softmax(dim=1)
context = torch.sum(image_features * weights.unsqueeze(-1), dim=1)

print("attention weights:", weights.shape)
print("sum per sample:", weights.sum(dim=1))
print("context:", context.shape)

In [ ]:
vocab_size = 100
embed_dim = 16
hidden_dim = 24

embedding = nn.Embedding(vocab_size, embed_dim)
lstm = nn.LSTMCell(embed_dim + D_img, hidden_dim)
output_head = nn.Linear(hidden_dim, vocab_size)

token = torch.tensor([1, 5])
token_emb = embedding(token)
h0 = torch.zeros(B, hidden_dim)
c0 = torch.zeros(B, hidden_dim)

h1, c1 = lstm(torch.cat([token_emb, context], dim=1), (h0, c0))
token_logits = output_head(h1)

print("token logits:", token_logits.shape)

## CLIP：图文对比学习

CLIP 将图像和文本投射到同一个 embedding 空间。

一个 batch 中：

- 对角线：正确图文配对；
- 非对角线：负样本。

训练目标就是让正确配对相似度高。

In [ ]:
B, D = 4, 64
img_emb = F.normalize(torch.randn(B, D), dim=1)
txt_emb = F.normalize(torch.randn(B, D), dim=1)

logit_scale = 10.0
logits = logit_scale * img_emb @ txt_emb.T
labels = torch.arange(B)

loss_i = F.cross_entropy(logits, labels)
loss_t = F.cross_entropy(logits.T, labels)
clip_loss = (loss_i + loss_t) / 2

print("similarity matrix:", logits.shape)
print("CLIP-style loss:", round(float(clip_loss), 4))

## CLIPCap：把视觉 embedding 变成“文本前缀”

核心思想：

`CLIP image embedding → mapping network → prefix embeddings → GPT-2`

这与后续多模态 LLM 中“视觉编码器 + projector/Q-Former + LLM”的思想高度相似。

In [ ]:
image_dim = 512
gpt_dim = 768
prefix_len = 10

mapping = nn.Linear(image_dim, prefix_len * gpt_dim)
clip_image_feature = torch.randn(2, image_dim)
prefix = mapping(clip_image_feature).view(2, prefix_len, gpt_dim)

print("CLIP feature:", clip_image_feature.shape)
print("GPT prefix embeddings:", prefix.shape)

# 8.8 图像检索（上）——理论基础

原教程顺序：

1. 图像检索系统
2. R@K / mAP
3. metric learning loss
4. Faiss 等向量检索框架
5. LSH / HNSW / PQ / IVF
6. 召回、粗排、精排、重排

检索系统的两个阶段：

- offline：建立 embedding database / index
- online：query 编码 → ANN search → rerank

## Recall@K 与 AP / mAP

- Recall@K：前 K 个结果中是否/有多少相关样本被召回；
- AP：相关结果越靠前越高；
- mAP：多个 query 的 AP 平均。

In [ ]:
def average_precision(ranked_ids, relevant_ids):
    relevant_ids = set(relevant_ids)
    hit = 0
    precisions = []
    for rank, item in enumerate(ranked_ids, start=1):
        if item in relevant_ids:
            hit += 1
            precisions.append(hit / rank)
    return sum(precisions) / max(len(relevant_ids), 1)

ranked = [3, 8, 2, 5, 9, 1]
relevant = {3, 2, 1}

ap = average_precision(ranked, relevant)
recall_at_3 = len(set(ranked[:3]) & relevant) / len(relevant)

print("AP:", round(ap, 3))
print("Recall@3:", round(recall_at_3, 3))

## Triplet Loss

目标：

$$
d(a,p)+m < d(a,n)
$$

即 anchor 更接近 positive，并与 negative 至少拉开 margin。

In [ ]:
triplet = nn.TripletMarginLoss(margin=0.5, p=2)

anchor = torch.tensor([[1.0, 0.0]])
positive = torch.tensor([[0.9, 0.1]])
negative = torch.tensor([[0.0, 1.0]])

loss_triplet = triplet(anchor, positive, negative)
print("triplet loss:", float(loss_triplet))

## 精确检索：归一化 + 点积 = cosine similarity

CLIP 类模型输出 embedding 后，通常先做 L2 normalization，再用 inner product / cosine similarity 检索。

In [ ]:
db = F.normalize(torch.randn(100, 32), dim=1)
query = F.normalize(torch.randn(1, 32), dim=1)

similarity = query @ db.T
values, indices = similarity.topk(k=5, dim=1)

print("top-5 ids:", indices[0].tolist())
print("top-5 cosine:", [round(v, 3) for v in values[0].tolist()])

## LSH / HNSW / PQ / IVF 应该抓住什么？

- **LSH**：哈希分桶，让近邻高概率进入同桶。
- **HNSW**：多层近邻图，从稀疏上层快速导航到局部区域。
- **PQ**：把高维向量切成子空间，各自量化，用 codebook 降内存与计算。
- **IVF**：先聚类；query 只搜索最相关的若干簇。
- **IVF+PQ**：先缩小搜索空间，再量化压缩。

关键权衡永远是：

> latency / memory / recall 三者之间做折中。

## 召回 → 粗排 → 精排 → 重排

大规模系统不会让最昂贵模型直接扫全库。

- recall：优先“不漏”，速度第一；
- coarse rank：快速初筛；
- fine rank：更精确、更昂贵；
- rerank：叠加业务规则、多样性、去重等。

# 8.8 图像检索（下）——CLIP + Faiss + Web 服务

原教程工程链路：

`图片库 → CLIP 编码 → Faiss 建索引 → query 编码 → top-k → id/path 映射 → Flask 展示`

本 Notebook 不下载 CLIP 和 Faiss，但下面用相同接口思想构造最小检索模块。

In [ ]:
class MiniRetrievalSystem:
    def __init__(self, db_embeddings, db_ids):
        self.db = F.normalize(db_embeddings, dim=1)
        self.db_ids = list(db_ids)

    @torch.inference_mode()
    def search(self, query_embedding, k=3):
        q = F.normalize(query_embedding, dim=1)
        scores = q @ self.db.T
        values, idx = scores.topk(k=k, dim=1)
        result_ids = [self.db_ids[i] for i in idx[0].tolist()]
        return result_ids, values[0]

system = MiniRetrievalSystem(
    db_embeddings=torch.randn(20, 64),
    db_ids=[f"img_{i:03d}.jpg" for i in range(20)]
)

ids, scores = system.search(torch.randn(1, 64), k=3)
print(ids)
print([round(float(x), 3) for x in scores])

### Faiss 对应关系

真实工程中常见：

```python
# cosine similarity
faiss.normalize_L2(database)
faiss.normalize_L2(query)

index = faiss.IndexFlatIP(dim)
index.add(database)
scores, ids = index.search(query, k)
```

若使用 IVF / PQ，需要先训练索引，再 `add()`。

### Web 层

Flask/FastAPI 只负责服务封装，不应承担模型和索引构建逻辑。更合理的结构：

- `FeatureEncoder`
- `VectorIndex`
- `RetrievalService`
- Web API

这样离线建库、在线查询、接口层可以独立测试。

# 章末知识结构总结

这一章表面上有很多任务，但可以压缩成四条主线：

### 1. 判别式视觉任务

- 分类：整图 → 类别
- 分割：像素 → 类别
- 检测：目标 → bbox + 类别
- 跟踪：跨帧 bbox → 稳定 identity

### 2. 生成模型

- CycleGAN：对抗训练 + 循环一致性
- DDPM：学习噪声预测，从噪声逐步生成图像

### 3. 视觉-语言

- CNN+RNN：视觉特征喂给语言解码器
- CLIP：图像与文本对齐到统一 embedding 空间
- CLIPCap：视觉 embedding 通过 mapping network 变成 LLM 可接受的 prefix

### 4. 检索系统

- encoder 决定 embedding 质量
- ANN index 决定规模与延迟
- rerank 决定最终排序质量
- normalization、metric、index 参数必须一致

# 学完必须会回答的 12 个问题

1. 图像分类中 `model.eval()` 与 `torch.inference_mode()` 分别解决什么问题？
2. 医学影像为什么应按患者划分训练/验证，而不能随机按切片划分？
3. 二分类分割中 `BCEWithLogitsLoss` 的输出和标签 shape / dtype 应是什么？
4. Dice 与 IoU 的数学关系和使用差异是什么？
5. VOC、COCO、YOLO 三种 bbox 格式如何转换？
6. NMS 为什么需要 IoU 阈值？阈值太大或太小会怎样？
7. DeepSORT 为什么同时需要外观特征、IoU/运动信息与 Hungarian Algorithm？
8. Kalman Filter 的 prediction、measurement、Kalman Gain 各自代表什么？
9. CycleGAN 为什么需要 cycle consistency loss？只用 GAN loss 会有什么问题？
10. DDPM 为什么训练时可以直接从 $x_0$ 得到任意 $x_t$，不需要逐步加噪？
11. CLIP 的对比学习中为什么 batch 相似度矩阵的对角线是正样本？
12. IVF、PQ、HNSW 分别主要在解决速度、内存还是图搜索问题？它们的 recall-latency tradeoff 如何理解？

## 建议的复习优先级

如果后续目标是 LLM / Transformer / 多模态，优先复习：

1. **DDPM 中 timestep embedding 与 attention**
2. **CLIP 对比学习**
3. **CLIPCap 的视觉 embedding → prefix mapping**
4. **向量归一化 + cosine similarity + ANN retrieval**
5. U-Net / ResBlock / skip connection

分类、检测、跟踪保留工程认知即可，不需要当前阶段深挖 YOLO 或 DeepSORT 源码。